# OpenPlaque — Combined Total Plaque Volume + Canonical RCA PCAT

One Colab workflow that runs the established OpenPlaque total plaque volume analysis on LAD/RCA/LCX curved reformats and the locked RCA 10–50 mm PCAT analysis, then creates consolidated summary tables of all metrics.

The TPV section follows the established main-branch workflow: raw nnU-Net plaque volume, boundary-refined working estimate, conservative eroded core, and core→raw uncertainty interval.

The PCAT section uses the frozen accepted RCA centerline/radius model and the locked shared circular sampler (+0.75 mm outer-wall margin; adipose −190 to −30 HU). It also recomputes the five circular geometry-sensitivity margins. Directional-interface sensitivity is included in the final tables when its saved result is available.

Research use only. Not for clinical decision-making. This is **OpenPlaque PCAT Attenuation**, not Caristo FAI-Score.


In [ ]:
# FIRST EXECUTABLE CELL — always mount Drive first.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Clone only this fresh main-derived branch and install the established Colab requirements.
!rm -rf /content/OpenPlaque
!git clone -q --branch combined-tpv-pcat-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
print('Repository and requirements ready.')


## 1. Configure OpenPlaque and nnU-Net


In [ ]:
import os, sys, shutil, zipfile, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy.spatial import cKDTree
from IPython.display import display, HTML

REPO = Path('/content/OpenPlaque')
sys.path.insert(0, str(REPO/'src'))
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUT = ROOT/'Combined_TPV_PCAT_All_Metrics'
OUT.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'
for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

model_zip = ROOT/'models'/'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')
if not model_target.exists():
    if not model_zip.exists():
        raise FileNotFoundError(f'Missing nnU-Net model ZIP: {model_zip}')
    with zipfile.ZipFile(model_zip) as zf:
        zf.extractall('/content/nnUNet_results')
print('Output folder:', OUT)
print('nnUNet results:', os.environ['nnUNet_results'])


## 2. Load the full DICOM study once

The Drive ZIP is copied locally first so the TPV and PCAT sections share the same local study without repeatedly reading compressed data from Drive.


In [ ]:
from openplaque.study import OpenPlaqueStudy

drive_zip = ROOT/'Full_DICOM.zip'
local_zip = Path('/content/Full_DICOM.zip')
if not drive_zip.exists():
    raise FileNotFoundError(drive_zip)
if not local_zip.exists() or local_zip.stat().st_size != drive_zip.stat().st_size:
    shutil.copyfile(drive_zip, local_zip)
extract_root = '/content/full_dicom_combined_metrics'
shutil.rmtree(extract_root, ignore_errors=True)
study = OpenPlaqueStudy(str(local_zip), extract_root=extract_root)
study.summary()


# Part A — Established total plaque volume analysis

This reproduces the established main-branch TPV workflow: automatic curved-series detection, nnU-Net segmentation of LAD/RCA/LCX, raw TPV, boundary-refined TPV, conservative core TPV, and the core→raw uncertainty interval.


In [ ]:
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series
from openplaque.uncertainty import make_tpv_uncertainty_summary

fallback_series = {'RCA':1035, 'LCX':1039, 'LAD':1043}
series_map, candidates = detect_artery_series(study, fallback_series=fallback_series, return_candidates=True)
print('Detected curved coronary series:', series_map)

loaded = {}
for vessel in ['LAD','RCA','LCX']:
    image, volume, files = study.load_series(series_map[vessel])
    loaded[vessel] = (image, volume, files)
    print(vessel, 'series', series_map[vessel], 'shape', volume.shape, 'spacing', image.GetSpacing())

reports = []
for vessel in ['LAD','RCA','LCX']:
    image, volume, _ = loaded[vessel]
    print('\nSegmenting', vessel, '...')
    report = segment_vessel(image, volume, vessel)
    reports.append(report)
    report.summary()


In [ ]:
# Established main estimate and conservative core.
def main_refinement(report):
    return refine_plaque_mask(
        volume=report.volume, mask=report.mask, spacing=report.mask_image.GetSpacing(),
        remove_small=True, min_component_voxels=10,
        trim_lumen_adjacent=True, lumen_distance_voxels=1,
        erode_core=False, high_hu_threshold=None, low_hu_threshold=None)

def conservative_core(report):
    return refine_plaque_mask(
        volume=report.volume, mask=report.mask, spacing=report.mask_image.GetSpacing(),
        remove_small=True, min_component_voxels=10,
        trim_lumen_adjacent=True, lumen_distance_voxels=1,
        erode_core=True, erosion_iterations=1,
        high_hu_threshold=None, low_hu_threshold=None)

refinements = {r.name: main_refinement(r) for r in reports}
core_results = {r.name: conservative_core(r) for r in reports}
uncertainty = make_tpv_uncertainty_summary(reports, refinements, core_results)
tpv_df = pd.DataFrame(uncertainty.rows())

# Add useful voxel counts and derived percentages.
report_map = {r.name:r for r in reports}
raw_vox=[]; refined_vox=[]; core_vox=[]
for vessel in tpv_df['vessel']:
    if vessel == 'TOTAL':
        raw_vox.append(sum(r.plaque_voxels for r in reports))
        refined_vox.append(sum(refinements[r.name].refined_plaque_voxels for r in reports))
        core_vox.append(sum(core_results[r.name].refined_plaque_voxels for r in reports))
    else:
        raw_vox.append(report_map[vessel].plaque_voxels)
        refined_vox.append(refinements[vessel].refined_plaque_voxels)
        core_vox.append(core_results[vessel].refined_plaque_voxels)
tpv_df['raw_plaque_voxels'] = raw_vox
tpv_df['refined_plaque_voxels'] = refined_vox
tpv_df['core_plaque_voxels'] = core_vox
tpv_df['refined_vs_raw_pct'] = 100.0*tpv_df.refined_tpv_mm3/tpv_df.raw_tpv_mm3.replace(0,np.nan)
tpv_df['removed_from_raw_pct'] = 100.0*(tpv_df.raw_tpv_mm3-tpv_df.refined_tpv_mm3)/tpv_df.raw_tpv_mm3.replace(0,np.nan)
tpv_df.to_csv(OUT/'tpv_metrics_by_vessel.csv', index=False)
display(tpv_df)


In [ ]:
# Save segmentation masks and one compact visual QC figure.
seg_out = OUT/'segmentations'; seg_out.mkdir(exist_ok=True)
fig, axs = plt.subplots(1,3,figsize=(15,5))
for ax, r in zip(axs, reports):
    sitk.WriteImage(r.mask_image, str(seg_out/f'{r.name}_raw_segmentation.nii.gz'))
    refined_img = sitk.GetImageFromArray(refinements[r.name].refined_mask.astype(np.uint8))
    refined_img.CopyInformation(r.mask_image)
    sitk.WriteImage(refined_img, str(seg_out/f'{r.name}_refined_segmentation.nii.gz'))
    counts = np.sum(refinements[r.name].refined_mask==2, axis=(1,2))
    z = int(np.argmax(counts)) if np.any(counts) else r.volume.shape[0]//2
    ax.imshow(r.volume[z], cmap='gray', vmin=-200, vmax=800)
    ax.contour(refinements[r.name].refined_mask[z]==2, levels=[0.5], linewidths=1)
    ax.set_title(f'{r.name} refined plaque')
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT/'01_tpv_refined_qc.png', dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)


# Part B — Locked RCA 10–50 mm PCAT analysis

This uses the frozen accepted RCA centerline and lumen-radius profile produced during PCAT validation. Centerline extraction itself is intentionally not rerun here; the purpose is to run the locked PCAT metric alongside the established TPV analysis.


In [ ]:
# Load source CCTA series 7 and the frozen RCA coordinate/radius model.
PCAT_BASE = ROOT/'PCAT_RCA_10_50'
cp = PCAT_BASE/'rca_centerline_smoothed_zyx.csv'
rp = PCAT_BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists():
    raise FileNotFoundError('Missing frozen PCAT centerline/radius inputs. Run the accepted RCA PCAT prototype first.')

source_img, ct, _ = study.load_series(7)
ct = np.asarray(ct)
sp_xyz = np.array(source_img.GetSpacing(), float)
sp_zyx = sp_xyz[::-1]
voxel_mm3 = float(np.prod(sp_xyz))

cl = pd.read_csv(cp)
rad = pd.read_csv(rp)
arc = cl.arc_mm.to_numpy(float)
pts_zyx = cl[['z','y','x']].to_numpy(float)
pts_mm = pts_zyx*sp_zyx
lumen_all = np.interp(arc, rad.arc_mm.to_numpy(float), rad.lumen_radius_mm.to_numpy(float))
SEG0, SEG1 = 10.0, 50.0
FAT_LO_HU, FAT_HI_HU = -190.0, -30.0
PRIMARY_MARGIN_MM = 0.75
MARGINS_MM = [0.25,0.50,0.75,1.00,1.25]
m = (arc>=SEG0)&(arc<=SEG1)
seg_arc = arc[m]; seg_zyx = pts_zyx[m]; seg_mm = pts_mm[m]; seg_lumen = lumen_all[m]
print('PCAT segment points:', len(seg_arc), 'arc range', seg_arc.min(), 'to', seg_arc.max(), 'mm')
print('Mean lumen radius:', round(float(seg_lumen.mean()),3), 'mm')


In [ ]:
# Build the one immutable voxel map used by every circular-wall PCAT calculation.
max_outer = float(np.max(seg_lumen + max(MARGINS_MM)))
max_shell_outer = 3.0*max_outer
pad_mm = max_shell_outer + 3.0
lo = np.floor(np.min(seg_zyx,axis=0)-pad_mm/sp_zyx).astype(int)
hi = np.ceil(np.max(seg_zyx,axis=0)+pad_mm/sp_zyx).astype(int)+1
lo = np.maximum(lo,0); hi = np.minimum(hi,np.array(ct.shape))
crop = ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
zz,yy,xx = np.indices(crop.shape)
g_zyx = np.stack([zz+lo[0],yy+lo[1],xx+lo[2]],axis=-1).reshape(-1,3).astype(float)
g_mm = g_zyx*sp_zyx
tree = cKDTree(seg_mm)
dist_mm, nearest_idx = tree.query(g_mm,k=1,workers=-1)
nearest_idx = nearest_idx.astype(int)
nearest_arc = seg_arc[nearest_idx]
nearest_lumen = seg_lumen[nearest_idx]
hu = crop.reshape(-1).astype(float)
fat_hu = (hu>=FAT_LO_HU)&(hu<=FAT_HI_HU)

aorta_candidates = [
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz']
ap = next((p for p in aorta_candidates if p.exists()), None)
if ap is None:
    raise FileNotFoundError('Canonical PCAT requires the cached TotalSegmentator aorta mask.')
ai = sitk.ReadImage(str(ap))
if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()):
    ai = sitk.Resample(ai, source_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
aorta = sitk.GetArrayFromImage(ai)>0
aorta_flat = aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].reshape(-1)
print('Immutable PCAT crop:', crop.shape, 'aorta mask:', ap)


In [ ]:
def compute_circular_pcat(wall_margin_mm):
    margin=float(wall_margin_mm)
    outer = nearest_lumen + margin
    shell_outer = 3.0*outer  # outward thickness = one local outer diameter
    shell = (dist_mm>outer)&(dist_mm<=shell_outer)&(nearest_arc>=SEG0)&(nearest_arc<=SEG1)&(~aorta_flat)
    fat = shell & fat_hu
    vals = hu[fat]
    if len(vals)==0:
        raise RuntimeError(f'No PCAT voxels for margin {margin}')
    radial_out = dist_mm-outer
    radial_rows=[]
    for b in np.arange(0.0,6.0,0.5):
        fm=fat&(radial_out>=b)&(radial_out<b+0.5); vv=hu[fm]
        radial_rows.append({'wall_margin_mm':margin,'radial_start_mm':b,'radial_end_mm':b+0.5,'fat_voxels':len(vv),'mean_hu':np.mean(vv) if len(vv) else np.nan})
    long_rows=[]
    for b in range(10,50):
        fm=fat&(nearest_arc>=b)&(nearest_arc<b+1); vv=hu[fm]
        long_rows.append({'wall_margin_mm':margin,'arc_start_mm':b,'arc_end_mm':b+1,'fat_voxels':len(vv),'mean_hu':np.mean(vv) if len(vv) else np.nan})
    return {'wall_margin_mm':margin,'pcat_mean_hu':float(np.mean(vals)),'pcat_median_hu':float(np.median(vals)),
            'pcat_sd_hu':float(np.std(vals)),'fat_voxels':int(fat.sum()),'fat_volume_ml':float(fat.sum()*voxel_mm3/1000.0),
            'shell_voxels':int(shell.sum()),'shell_volume_ml':float(shell.sum()*voxel_mm3/1000.0),
            'fat_fraction':float(fat.sum()/max(1,shell.sum())),'radial':pd.DataFrame(radial_rows),'longitudinal':pd.DataFrame(long_rows)}

primary = compute_circular_pcat(PRIMARY_MARGIN_MM)
sens_results = [compute_circular_pcat(x) for x in MARGINS_MM]
same = next(x for x in sens_results if x['wall_margin_mm']==PRIMARY_MARGIN_MM)
assert primary['pcat_mean_hu']==same['pcat_mean_hu']
assert primary['fat_voxels']==same['fat_voxels']
assert primary['shell_voxels']==same['shell_voxels']
pcat_sensitivity_df = pd.DataFrame([{k:v for k,v in x.items() if k not in ['radial','longitudinal']} for x in sens_results])
pcat_primary_df = pd.DataFrame([{k:v for k,v in primary.items() if k not in ['radial','longitudinal']}])
pcat_primary_df.to_csv(OUT/'pcat_canonical_primary.csv',index=False)
pcat_sensitivity_df.to_csv(OUT/'pcat_circular_sensitivity.csv',index=False)
primary['radial'].to_csv(OUT/'pcat_canonical_radial.csv',index=False)
primary['longitudinal'].to_csv(OUT/'pcat_canonical_longitudinal.csv',index=False)
display(pcat_primary_df)
display(pcat_sensitivity_df)


In [ ]:
# Optional directional-interface sensitivity result from the completed validation experiment.
directional = None
dpath = ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_summary.csv'
if dpath.exists():
    ddf = pd.read_csv(dpath)
    directional = ddf.iloc[0].to_dict()
    print('Loaded directional-interface sensitivity:', dpath)
else:
    print('Directional-interface summary not found; circular canonical analysis is complete without it.')

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(pcat_sensitivity_df.wall_margin_mm, pcat_sensitivity_df.pcat_mean_hu, marker='o', label='circular sensitivity')
ax.axhline(primary['pcat_mean_hu'], linestyle='--', label='canonical +0.75 mm')
if directional is not None and 'directional_pcat_mean_hu' in directional:
    ax.axhline(float(directional['directional_pcat_mean_hu']), linestyle=':', label='directional interface')
ax.set_xlabel('outer-wall margin (mm)'); ax.set_ylabel('PCAT mean HU'); ax.set_title('RCA PCAT geometry sensitivity'); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'02_pcat_geometry_sensitivity.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


# Final consolidated summary tables


In [ ]:
# Table 1: TPV by vessel and total.
print('TABLE 1 — TOTAL PLAQUE VOLUME / UNCERTAINTY')
display(tpv_df.round(3))

# Table 2: canonical PCAT metrics.
pcat_metrics = pd.DataFrame([
    ['RCA segment start',SEG0,'mm'],['RCA segment end',SEG1,'mm'],['fat HU lower bound',FAT_LO_HU,'HU'],['fat HU upper bound',FAT_HI_HU,'HU'],
    ['outer-wall margin',PRIMARY_MARGIN_MM,'mm'],['mean lumen radius',float(seg_lumen.mean()),'mm'],
    ['PCAT mean',primary['pcat_mean_hu'],'HU'],['PCAT median',primary['pcat_median_hu'],'HU'],['PCAT SD',primary['pcat_sd_hu'],'HU'],
    ['PCAT fat voxels',primary['fat_voxels'],'voxels'],['PCAT fat volume',primary['fat_volume_ml'],'mL'],
    ['PCAT shell voxels',primary['shell_voxels'],'voxels'],['PCAT shell volume',primary['shell_volume_ml'],'mL'],['PCAT shell fat fraction',primary['fat_fraction'],'fraction']
],columns=['metric','value','unit'])
print('TABLE 2 — CANONICAL RCA PCAT')
display(pcat_metrics)

# Table 3: geometry sensitivity, including directional-interface if available.
geometry = pcat_sensitivity_df[['wall_margin_mm','pcat_mean_hu','pcat_median_hu','pcat_sd_hu','fat_voxels','fat_volume_ml','shell_volume_ml','fat_fraction']].copy()
geometry['method'] = geometry.wall_margin_mm.map(lambda x:f'circular +{x:.2f} mm')
if directional is not None and 'directional_pcat_mean_hu' in directional:
    geometry = pd.concat([geometry,pd.DataFrame([{'wall_margin_mm':np.nan,'pcat_mean_hu':float(directional['directional_pcat_mean_hu']),
        'pcat_median_hu':directional.get('directional_pcat_median_hu',np.nan),'pcat_sd_hu':directional.get('directional_pcat_sd_hu',np.nan),
        'fat_voxels':directional.get('fat_voxels',np.nan),'fat_volume_ml':directional.get('fat_volume_ml',np.nan),
        'shell_volume_ml':directional.get('shell_volume_ml',np.nan),'fat_fraction':np.nan,'method':'directional fat-interface'}])],ignore_index=True)
geometry['delta_from_canonical_hu'] = geometry.pcat_mean_hu-primary['pcat_mean_hu']
geometry.to_csv(OUT/'pcat_geometry_method_comparison.csv',index=False)
print('TABLE 3 — PCAT GEOMETRY SENSITIVITY')
display(geometry.round(3))


In [ ]:
# Table 4: one-row subject-level summary and a long-form all-metrics table.
tot = tpv_df.loc[tpv_df.vessel=='TOTAL'].iloc[0]
vessel_ref = {r.vessel:float(r.refined_tpv_mm3) for _,r in tpv_df[tpv_df.vessel!='TOTAL'].iterrows()}
method_means = geometry.pcat_mean_hu.dropna().to_numpy(float)
subject_summary = pd.DataFrame([{'total_raw_tpv_mm3':float(tot.raw_tpv_mm3),
    'total_refined_tpv_mm3':float(tot.refined_tpv_mm3),'total_core_tpv_mm3':float(tot.core_tpv_mm3),
    'total_tpv_uncertainty_width_mm3':float(tot.uncertainty_width_mm3),'LAD_refined_tpv_mm3':vessel_ref.get('LAD',np.nan),
    'RCA_refined_tpv_mm3':vessel_ref.get('RCA',np.nan),'LCX_refined_tpv_mm3':vessel_ref.get('LCX',np.nan),
    'RCA_PCAT_mean_HU':primary['pcat_mean_hu'],'RCA_PCAT_median_HU':primary['pcat_median_hu'],'RCA_PCAT_SD_HU':primary['pcat_sd_hu'],
    'RCA_PCAT_fat_volume_ml':primary['fat_volume_ml'],'RCA_PCAT_geometry_min_HU':float(np.min(method_means)),
    'RCA_PCAT_geometry_max_HU':float(np.max(method_means)),'RCA_PCAT_max_abs_delta_HU':float(np.max(np.abs(method_means-primary['pcat_mean_hu']))),
    'RCA_PCAT_segment_start_mm':SEG0,'RCA_PCAT_segment_end_mm':SEG1,'RCA_PCAT_wall_margin_mm':PRIMARY_MARGIN_MM}])
subject_summary.to_csv(OUT/'subject_summary_all_metrics.csv',index=False)
print('TABLE 4 — ONE-ROW ALL-METRICS SUMMARY')
display(subject_summary.T)

rows=[]
for _,r in tpv_df.iterrows():
    for col,unit in [('raw_tpv_mm3','mm3'),('refined_tpv_mm3','mm3'),('core_tpv_mm3','mm3'),('removed_boundary_mm3','mm3'),('uncertainty_width_mm3','mm3')]:
        rows.append({'category':'TPV','scope':r.vessel,'metric':col,'value':float(r[col]),'unit':unit})
for _,r in pcat_metrics.iterrows():
    rows.append({'category':'PCAT canonical','scope':'RCA 10-50 mm','metric':r.metric,'value':float(r.value),'unit':r.unit})
for _,r in geometry.iterrows():
    rows.append({'category':'PCAT sensitivity','scope':r['method'],'metric':'pcat_mean_hu','value':float(r.pcat_mean_hu),'unit':'HU'})
all_metrics = pd.DataFrame(rows)
all_metrics.to_csv(OUT/'all_metrics_long.csv',index=False)


In [ ]:
# Create a readable HTML summary and one report-back ZIP.
html = ['<html><head><meta charset="utf-8"><title>OpenPlaque Combined TPV + PCAT</title></head><body>',
        '<h1>OpenPlaque Combined TPV + Canonical RCA PCAT</h1>',
        '<p>Research use only. PCAT metric is OpenPlaque PCAT Attenuation, not Caristo FAI-Score.</p>',
        '<h2>TPV by vessel</h2>',tpv_df.round(3).to_html(index=False),
        '<h2>Canonical RCA PCAT</h2>',pcat_metrics.round(4).to_html(index=False),
        '<h2>PCAT geometry sensitivity</h2>',geometry.round(3).to_html(index=False),
        '<h2>One-row all-metrics summary</h2>',subject_summary.round(4).to_html(index=False),'</body></html>']
html_path = OUT/'OPENPLAQUE_COMBINED_TPV_PCAT_SUMMARY.html'
html_path.write_text('\n'.join(html),encoding='utf-8')

zip_path = OUT/'OPENPLAQUE_COMBINED_TPV_PCAT_REPORT_BACK.zip'
files = [OUT/'tpv_metrics_by_vessel.csv',OUT/'pcat_canonical_primary.csv',OUT/'pcat_circular_sensitivity.csv',
         OUT/'pcat_canonical_radial.csv',OUT/'pcat_canonical_longitudinal.csv',OUT/'pcat_geometry_method_comparison.csv',
         OUT/'subject_summary_all_metrics.csv',OUT/'all_metrics_long.csv',OUT/'01_tpv_refined_qc.png',
         OUT/'02_pcat_geometry_sensitivity.png',html_path]
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        if p.exists(): zf.write(p,arcname=p.name)
print('\nDONE. Outputs:',OUT)
print('Report-back ZIP:',zip_path)
print('Direct Drive search URLs:')
print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_COMBINED_TPV_PCAT_REPORT_BACK.zip')
print('https://drive.google.com/drive/u/0/search?q=subject_summary_all_metrics.csv')
print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_COMBINED_TPV_PCAT_SUMMARY.html')
